# 03 - Verdict model
Sentence selection (bi-encoder vs cross-encoder reranker), NLI scoring, and threshold tuning.

Thresholds are tuned on the **train** split and reported on **validation**. The NLI model is zero-shot, so train claims are unseen data for it, and the validation number stays honest. Scoring takes a while on CPU; use a Colab GPU if you can.

In [ ]:
import os, sys, subprocess
if "google.colab" in sys.modules:  # Colab: clone the repo and install dependencies
    if not os.path.isdir("fact-and-logic"):
        subprocess.run(["git", "clone", "https://github.com/Anishkumar25/fact-and-logic.git"], check=True)
    os.chdir("fact-and-logic")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
import pandas as pd
from src import data, evaluate
from src.config import ART
from src.retrieval import Retriever
from src.verdict import Verifier

r = Retriever()
train, val = data.claims('train'), data.claims('validation')

## 1. Does the reranker pick better evidence sentences?
Share of claims where a gold sentence is in the top-5 selected sentences.

In [ ]:
recall = {name: evaluate.sentence_recall(Verifier(r, reranker=rr), val) for name, rr in [('bi-encoder', False), ('cross-encoder', True)]}
recall

## 2. Score every claim once, then tune cheaply
Set `N = 100` for a quick smoke run.

In [ ]:
N = None
tr, va = (train.head(N), val.head(N)) if N else (train, val)
v = Verifier(r, reranker=True)
st = evaluate.score_all(v, tr, ART / 'nb_train.parquet')
sv = evaluate.score_all(v, va, ART / 'nb_val.parquet')
best = evaluate.tune(st, tr.label)
best

In [ ]:
rep = {'default 0.5': evaluate.report(sv, va.label, .5, .5), 'tuned on train': evaluate.report(sv, va.label, best['ts'], best['tc'])}
pd.DataFrame({k: {'macro_f1': x['macro_f1'], 'accuracy': x['accuracy']} for k, x in rep.items()}).T.round(3)

## 3. Confusion matrix and errors

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
from src.config import LABELS
from src.verdict import decide

pred = [decide(a, b, best['ts'], best['tc']) for a, b in zip(sv.sup, sv.con)]
ConfusionMatrixDisplay.from_predictions(va.label, pred, labels=list(LABELS), xticks_rotation=20)

In [ ]:
wrong = [(c, y, p) for c, y, p in zip(va.claim, va.label, pred) if y != p]
for c, y, p in wrong[:10]:
    print(f'gold={y:16} pred={p:16} {c}')

## 4. Save thresholds for deployment
Run only after scoring the full train and validation sets (`N = None`). The API and app read `thresholds.json` at startup.

In [ ]:
import json
from src.config import THRESHOLDS
THRESHOLDS.write_text(json.dumps({'ts': best['ts'], 'tc': best['tc']}, indent=2))
THRESHOLDS.read_text()